<a href="https://colab.research.google.com/github/acedesci/scanalytics/blob/master/EN/S10_Prescriptive_Analytics/Module2_Retail_Price_Optimization_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook for Module 2: Implicit (compact) model of Rue La La

---
*Created by Yossiri Adulyasak*

---

Unlike the explicit model in which we need to explicitly add each complete equation one by one, we can automate the model generation process by using a script version of it. For this one, we do not expect that you understand in detail how to generate the script but simply understand what each block does. Creating the script would require some experience. The main purpose of this is to provide an example of real-life mathematical programming workflow which automate the prescriptive analytics process.

More particularly, we want to create an **implicit (or compact) model** of the following prescriptive pricing model of Rue La La.

![alt text](https://github.com/acedesci/scanalytics/blob/master/EN/S08_09_Retail_Analytics/implicit_model.png?raw=true)

The following blocks install Pyomo and solver. We also provide an option to use a more powerful solver *CBC* in addition to GLPK we used earlier. You can outcomment it if you want to switch to CBC.

In [1]:
# Install Pyomo and GLPK
!pip install -q pyomo
!apt-get install -y -qq glpk-utils #if GLPK is used
# !apt-get install -y -qq coinor-cbc #if cbc is used

Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package glpk-utils.
Preparing to unpack .../glpk-utils_5.0-1_amd64.deb ...
Unpacking glpk-utils (5.0-1) ...
Setting up libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4b

# Block 1: Data input

We prepared the data inputs in two files, i.e.,


1.   **'predictedSales_Prob1.csv'**. This is a small scale problem. It is identical to the problem you see in the simplified notebook for the explicit model.
2.   **'predictedSales_Prob2.csv'**. This is a large-scale problem. This one contains a much higher number of variables and constraints to reflect real-life setting.

Please mainly focus on the file *'predictedSales_Prob1.csv'* since you will get to see the same model as Module_1A. You can also try *'predictedSales_Prob2.csv'* if you are interested to see the large-scale model.

In order to read the input, we take the file from the URL. This is the same file that you would obtain if you run the module 1B. If you want, you can change this block so that you can upload it from your PC or load it from Google Drive (see also Module 1B how these two options can be done).



In [3]:
import pandas

# Prob1 is the same problem as Module 2A
predDemand = pandas.read_csv('https://raw.githubusercontent.com/acedesci/scanalytics/master/EN/S10_Prescriptive_Analytics/predictedSales_Prob1.csv')

# Prob2 is the large-scale problem
# predDemand = pandas.read_csv('https://raw.githubusercontent.com/acedesci/scanalytics/master/EN/S10_Prescriptive_Analytics/predictedSales_Prob2.csv')

# Dataset is now stored in a Pandas Dataframe predDemand
predDemand

,Unnamed: 0.1,Unnamed: 0,avgPriceChoice,UPC,PRICE,FEATURE,DISPLAY,TPR_ONLY,RELPRICE,predictSales
0,0,0,3,1600027528,2.5,0,0,0,0.833333,89.0
1,1,1,3,1600027528,3.0,0,0,0,1.000000,74.5
2,2,2,3,1600027528,3.5,0,0,0,1.166667,60.1
3,3,3,3,1600027564,2.5,0,0,0,0.833333,23.3
4,4,4,3,1600027564,3.0,0,0,0,1.000000,22.3
5,5,5,3,1600027564,3.5,0,0,0,1.166667,21.4
6,6,6,3,3000006340,2.5,0,0,0,0.833333,5.5
7,7,7,3,3000006340,3.0,0,0,0,1.000000,4.6
8,8,8,3,3000006340,3.5,0,0,0,1.166667,3.7
9,9,9,3,3800031829,2.5,0,0,0,0.833333,27.7


With the new dataset, we first need to check how many average price values are there because we need to run the optimization model for each value of the average price.

In [4]:
avgPriceList = predDemand['avgPriceChoice'].unique()
inputColumns = ['avgPriceChoice', 'UPC', 'PRICE','predictSales']
print("Possible average price choices (k/N.Product):"+str(avgPriceList))


Possible average price choices (k/N.Product):[3]


# Block 2: Prepare input parameters for the model

We can choose which value of $k$ we want to use in the optimization model from the *avgPriceChoice* we have in the dataset. In *'predictedSales_Prob1.csv'*, there is only one average price choice at $3.0 whereas in *'predictedSales_Prob2.csv'* there are 4 different price choices you can choose form.

If you want to try different average price choices, we would need to repeat this procedure for each average price value and record the corresponding optimal solution to decide how each product should be priced and at which average price level to generate the optimal revenue.

Note that in this demo, we use $p_{ij}$ instead of $p_{j}$ since it is easier to prepare the script but the model remains identical to the original model because $p_{ij} = p_{j},i=1,...,n$.

In [5]:
# Nere we choose which value of k (avgPriceValue x N. of products) we would like to use in the model
# Note that k must be among the choices where the prediction has been prepared
avgPriceValue =  avgPriceList[0]

# Now we select only the row which corresponds to the previously chosen value of avgPriceValue (again k = avgPriceValue x N. of products)
predDemand_k = predDemand.loc[predDemand['avgPriceChoice'] == avgPriceValue][inputColumns]
print(predDemand_k)
productList = predDemand_k['UPC'].unique()
priceList = predDemand_k['PRICE'].unique()

# Here we prepare the dictionary to be used in the optimization model
p = {}
D = {}

for upc in productList:
  for price in priceList:
    p[(upc,price)] = price
    D[(upc,price)] = predDemand_k.loc[(predDemand['UPC'] == upc) & (predDemand_k['PRICE'] == price)]['predictSales'].values[0]

print(p)
print(D)

    avgPriceChoice         UPC  PRICE  predictSales
0                3  1600027528    2.5          89.0
1                3  1600027528    3.0          74.5
2                3  1600027528    3.5          60.1
3                3  1600027564    2.5          23.3
4                3  1600027564    3.0          22.3
5                3  1600027564    3.5          21.4
6                3  3000006340    2.5           5.5
7                3  3000006340    3.0           4.6
8                3  3000006340    3.5           3.7
9                3  3800031829    2.5          27.7
10               3  3800031829    3.0          24.5
11               3  3800031829    3.5          21.3
{(np.int64(1600027528), np.float64(2.5)): np.float64(2.5), (np.int64(1600027528), np.float64(3.0)): np.float64(3.0), (np.int64(1600027528), np.float64(3.5)): np.float64(3.5), (np.int64(1600027564), np.float64(2.5)): np.float64(2.5), (np.int64(1600027564), np.float64(3.0)): np.float64(3.0), (np.int64(1600027564), np.float64

# Block 3: Create an optimization model

### Block 3.1: Variable declarations

Unlike the first part of today's session, we index the decision variables and demand parameters by the product and the price themselves rather than their index. Indeed, we previously denoted $x_{ij}=1$ if the price option $j$ is chosen for product $i$, and 0 otherwise. Now, our variable is denoted by $x_{1600027528,\ 3.0}=1$, which means that product UPC '1600027528' will be sold at 3.0 dollars. The same notational remark applies to predicted demand ($D_{ijk}$) for the sum of prices $k$ and price ($p_{ij}$) parameters. We can declare the constraint sets first (model.PriceChoiceUPC, model.sumPrice) and then **add** the constraint functions later.

In [6]:
from pyomo.environ import *

iIndexList = list(range(len(productList)))
jIndexList = list(range(len(priceList)))

model = ConcreteModel()
# Variables
model.x = Var(productList, priceList, within = Binary)

# Constraints
model.PriceChoiceUPC = ConstraintList()
model.sumPrice = ConstraintList()

# Print to review the model (equations are still not included)
model.pprint()

1 Var Declarations
    x : Size=12, Index={1600027528, 3000006340, 1600027564, 3800031829}*{3.5, 2.5, 3.0}
        Key               : Lower : Value : Upper : Fixed : Stale : Domain
        (1600027528, 2.5) :     0 :  None :     1 : False :  True : Binary
        (1600027528, 3.0) :     0 :  None :     1 : False :  True : Binary
        (1600027528, 3.5) :     0 :  None :     1 : False :  True : Binary
        (1600027564, 2.5) :     0 :  None :     1 : False :  True : Binary
        (1600027564, 3.0) :     0 :  None :     1 : False :  True : Binary
        (1600027564, 3.5) :     0 :  None :     1 : False :  True : Binary
        (3000006340, 2.5) :     0 :  None :     1 : False :  True : Binary
        (3000006340, 3.0) :     0 :  None :     1 : False :  True : Binary
        (3000006340, 3.5) :     0 :  None :     1 : False :  True : Binary
        (3800031829, 2.5) :     0 :  None :     1 : False :  True : Binary
        (3800031829, 3.0) :     0 :  None :     1 : False :  True : 

### Block 3.2: Adding an objective function

Instead of iteratively entering the value for each price and predicted sales, we can simply create a loop **for** each product and a loop **for** each price. The code now looks very much like the general equation $\sum_{i} \sum_{j} p_{ij} \cdot D_{ijk} \cdot x_{ij}$ we saw in the first part of today's session with some minor changes for notational simplification.

In [7]:
# Objective function

obj_expr = sum(p[(i,j)]*D[(i,j)]*model.x[i,j] for i in productList for j in priceList)
print(obj_expr)
model.OBJ = Objective(expr = obj_expr, sense = maximize)

222.5*x[1600027528,2.5] + 223.5*x[1600027528,3.0] + 210.35*x[1600027528,3.5] + 58.25*x[1600027564,2.5] + 66.9*x[1600027564,3.0] + 74.89999999999999*x[1600027564,3.5] + 13.75*x[3000006340,2.5] + 13.799999999999999*x[3000006340,3.0] + 12.950000000000001*x[3000006340,3.5] + 69.25*x[3800031829,2.5] + 73.5*x[3800031829,3.0] + 74.55*x[3800031829,3.5]


### Block 3.3: Adding constraints

**Constraint 1: One price choice must be selected for each product**

Similarly, we can create a loop to **add** constraint functions to the constraint set **for** each product to ensure that only one price on the list is selected for that product. Unlike the first part of today's session, we need not iteratively type each constraint.

In [8]:
# Constraints #1
for i in productList:
  const1_expr = sum(model.x[i,j] for j in priceList) == 1
  print(const1_expr)
  model.PriceChoiceUPC.add(expr = const1_expr)


x[1600027528,2.5] + x[1600027528,3.0] + x[1600027528,3.5]  ==  1
x[1600027564,2.5] + x[1600027564,3.0] + x[1600027564,3.5]  ==  1
x[3000006340,2.5] + x[3000006340,3.0] + x[3000006340,3.5]  ==  1
x[3800031829,2.5] + x[3800031829,3.0] + x[3800031829,3.5]  ==  1


**Constraint 2: The sum of the prices of all products must equal $k$**

Similar **for** loops apply to the average price constraint. Please refer to the first part of today's session for detailed elaboration.

In [9]:
# Constraints #2
const2_expr = sum(p[i,j]*model.x[i,j] for i in productList for j in priceList) == avgPriceValue*len(productList)
print(const2_expr)
model.sumPrice.add(expr = const2_expr)



2.5*x[1600027528,2.5] + 3.0*x[1600027528,3.0] + 3.5*x[1600027528,3.5] + 2.5*x[1600027564,2.5] + 3.0*x[1600027564,3.0] + 3.5*x[1600027564,3.5] + 2.5*x[3000006340,2.5] + 3.0*x[3000006340,3.0] + 3.5*x[3000006340,3.5] + 2.5*x[3800031829,2.5] + 3.0*x[3800031829,3.0] + 3.5*x[3800031829,3.5]  ==  12


We can print the model to review prior to solving it.

In [10]:
model.pprint()

1 Var Declarations
    x : Size=12, Index={1600027528, 3000006340, 1600027564, 3800031829}*{3.5, 2.5, 3.0}
        Key               : Lower : Value : Upper : Fixed : Stale : Domain
        (1600027528, 2.5) :     0 :  None :     1 : False :  True : Binary
        (1600027528, 3.0) :     0 :  None :     1 : False :  True : Binary
        (1600027528, 3.5) :     0 :  None :     1 : False :  True : Binary
        (1600027564, 2.5) :     0 :  None :     1 : False :  True : Binary
        (1600027564, 3.0) :     0 :  None :     1 : False :  True : Binary
        (1600027564, 3.5) :     0 :  None :     1 : False :  True : Binary
        (3000006340, 2.5) :     0 :  None :     1 : False :  True : Binary
        (3000006340, 3.0) :     0 :  None :     1 : False :  True : Binary
        (3000006340, 3.5) :     0 :  None :     1 : False :  True : Binary
        (3800031829, 2.5) :     0 :  None :     1 : False :  True : Binary
        (3800031829, 3.0) :     0 :  None :     1 : False :  True : 

# Block 4: Solution and results

Finally, we call the solver and obtain the optimal solution. We can see that product '1600027528' and '3000006340' both are also sold at price $\$2.5$, product '1600027564' and '3800031829' both  at price $\$3.5$ while the optimal objective value is $\$385.7$. The solution (values of $x$) is the same as in the explicit model.

In [11]:
# Solve the model
opt = SolverFactory('glpk')
opt.solve(model)

model.display()

Model unknown

  Variables:
    x : Size=12, Index={1600027528, 3000006340, 1600027564, 3800031829}*{3.5, 2.5, 3.0}
        Key               : Lower : Value : Upper : Fixed : Stale : Domain
        (1600027528, 2.5) :     0 :   1.0 :     1 : False : False : Binary
        (1600027528, 3.0) :     0 :   0.0 :     1 : False : False : Binary
        (1600027528, 3.5) :     0 :   0.0 :     1 : False : False : Binary
        (1600027564, 2.5) :     0 :   0.0 :     1 : False : False : Binary
        (1600027564, 3.0) :     0 :   0.0 :     1 : False : False : Binary
        (1600027564, 3.5) :     0 :   1.0 :     1 : False : False : Binary
        (3000006340, 2.5) :     0 :   1.0 :     1 : False : False : Binary
        (3000006340, 3.0) :     0 :   0.0 :     1 : False : False : Binary
        (3000006340, 3.5) :     0 :   0.0 :     1 : False : False : Binary
        (3800031829, 2.5) :     0 :   0.0 :     1 : False : False : Binary
        (3800031829, 3.0) :     0 :   0.0 :     1 : False :